# Optional raw image inventory

Adapted from the original preprocessing notebook. This is exploratory only: `01_prepare_and_audit.ipynb` creates the canonical manifests. Configure paths before running. Raw image displays are local, not committed outputs.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
INPUT_ROOT = "/kaggle/input/datasets/"

print(os.listdir(INPUT_ROOT))

In [ ]:
DATASET_1 = "/kaggle/input/datasets/jtiptj/chest-xray-pneumoniacovid19tuberculosis"
DATASET_2 = "//kaggle/input/datasets/sachinkumar413/covid-pneumonia-normal-chest-xray-images"
DATASET_3 = "/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database"
DATASET_4 = "/kaggle/input/datasets/offeibekoe/dataset-of-tuberculosis-chest-x-rays-images/Dataset of Tuberculosis Chest X-rays Images"

In [ ]:
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg")


def collect_images(folder, disease, source, split=None):
    records = []

    if not os.path.exists(folder):
        raise FileNotFoundError(folder)

    for filename in sorted(os.listdir(folder)):

        if filename.lower().endswith(IMAGE_EXTENSIONS):

            records.append({
                "path": os.path.join(folder, filename),
                "filename": filename,
                "disease": disease,
                "source": source,
                "original_split": split
            })

    return records

## Dataset 1

In [ ]:
records = []

for split in ["train", "val", "test"]:

    split_dir = os.path.join(DATASET_1, split)

    records += collect_images(
        os.path.join(split_dir, "COVID19"),
        disease="COVID",
        source="dataset_1",
        split=split
    )

    records += collect_images(
        os.path.join(split_dir, "NORMAL"),
        disease="NORMAL",
        source="dataset_1",
        split=split
    )

    records += collect_images(
        os.path.join(split_dir, "PNEUMONIA"),
        disease="PNEUMONIA",
        source="dataset_1",
        split=split
    )

    records += collect_images(
        os.path.join(split_dir, "TURBERCULOSIS"),
        disease="TUBERCULOSIS",
        source="dataset_1",
        split=split
    )

## Dataset 2

In [ ]:
records += collect_images(
    os.path.join(DATASET_2, "COVID"),
    disease="COVID",
    source="dataset_2"
)

records += collect_images(
    os.path.join(DATASET_2, "NORMAL"),
    disease="NORMAL",
    source="dataset_2"
)

records += collect_images(
    os.path.join(DATASET_2, "PNEUMONIA"),
    disease="PNEUMONIA",
    source="dataset_2"
)

## Dataset 3

In [ ]:
dataset3_root = os.path.join(
    DATASET_3,
    "COVID-19_Radiography_Dataset"
)

records += collect_images(
    os.path.join(dataset3_root, "COVID", "images"),
    disease="COVID",
    source="dataset_3"
)

records += collect_images(
    os.path.join(dataset3_root, "Normal", "images"),
    disease="NORMAL",
    source="dataset_3"
)

records += collect_images(
    os.path.join(dataset3_root, "Viral Pneumonia", "images"),
    disease="PNEUMONIA",
    source="dataset_3"
)

Dataset 4

In [ ]:
records += collect_images(
    os.path.join(DATASET_4,"Normal Chest X-rays"),
    disease = "NORMAL",
    source = "dataset_4"
)

records += collect_images(
    os.path.join(DATASET_4,"TB Chest X-rays"),
    disease = "TUBERCULOSIS",
    source = "dataset_4"
)

In [ ]:
df = pd.DataFrame(records)

print("Total images:", len(df))

df.tail()

Disease Class Counts

In [ ]:
df["disease"].value_counts()

Source Counts

In [ ]:
df["source"].value_counts()

In [ ]:
pd.crosstab(
    df["source"],
    df["disease"]
)

Check for missing files

In [ ]:
def get_image_size(path):
    with Image.open(path) as img:
        return img.size

In [ ]:
sizes = df["path"].apply(get_image_size)

df["width"] = sizes.apply(lambda x: x[0])
df["height"] = sizes.apply(lambda x: x[1])

In [ ]:
df.groupby("source")[["width", "height"]].describe()

In [ ]:
def get_image_mode(path):
    with Image.open(path) as img:
        return img.mode

In [ ]:
df["mode"] = df["path"].apply(get_image_mode)

pd.crosstab(df["source"], df["mode"])

In [ ]:
import matplotlib.pyplot as plt


def show_samples(df, source, disease, n=5):

    subset = df[
        (df["source"] == source) &
        (df["disease"] == disease)
    ]

    if subset.empty:
        raise ValueError("No images for the requested source/class")

    sample = subset.sample(
        min(n, len(subset)),
        random_state=SEED
    )

    plt.figure(figsize=(15, 3))

    for i, (_, row) in enumerate(sample.iterrows()):

        img = Image.open(row["path"])

        plt.subplot(1, len(sample), i + 1)
        plt.imshow(img, cmap="gray")
        plt.axis("off")
        plt.title(
            f"{source}\n{disease}"
        )

    plt.show()

In [ ]:
show_samples(df, "dataset_1", "COVID")

In [ ]:
show_samples(df, "dataset_2", "COVID")

In [ ]:
show_samples(df, "dataset_3", "COVID")

In [ ]:
OUTPUT_PATH = "/kaggle/working/cbmir_metadata.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved to:", OUTPUT_PATH)